# 🇧🇩 bangla-render — Kaggle / Colab Demo

This notebook demonstrates **bangla-render** on Kaggle and Google Colab.

It handles:
- Headless Qt setup (no display available)
- Bengali font installation and registration
- Line plot, heatmap, and confusion matrix examples

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/mbs57/bangla-render/main/examples/bangla_render_kaggle_colab_demo.ipynb)  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbs57/bangla-render/blob/main/examples/bangla_render_kaggle_colab_demo.ipynb)

---

## Step 1 — Install bangla-render

In [ ]:
!pip install bangla-render -q

## Step 2 — Headless Qt setup

Kaggle and Colab have no display. We must set `QT_QPA_PLATFORM=offscreen` **before** importing bangla_render.

In [ ]:
import os
os.environ["QT_QPA_PLATFORM"] = "offscreen"

## Step 3 — Install Bengali font and register with Qt

In [ ]:
import subprocess
import glob
import matplotlib.font_manager

# Install Noto fonts
subprocess.run(["apt-get", "install", "-y", "fonts-noto"], 
               capture_output=True)

# Rebuild font cache
subprocess.run(["fc-cache", "-fv"], capture_output=True)

# Reload matplotlib font manager
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

print("Font installation done.")

## Step 4 — Import and initialise bangla-render

In [ ]:
import bangla_render as br

# Initialise Qt in headless/offscreen mode
br.init_renderer()

# Check environment
env = br.check_environment()
print("Environment:", env)
print("Qt available:", env["qt_available"])
print("Kaggle:", env["kaggle"])
print("Colab:", env["colab"])
print("Headless:", env["headless"])

## Step 5 — Register Bengali font with Qt

On Kaggle/Colab, Qt cannot auto-discover system fonts.
We register the font file directly.

In [ ]:
# Find NotoSansBengali font file
noto_files = glob.glob(
    "/usr/share/fonts/**/NotoSansBengali-Regular.ttf",
    recursive=True
)

if noto_files:
    br.register_font(noto_files[0])
    br.set_default_font(font_family="Noto Sans Bengali")
    print("✅ Font registered:", br.get_default_font())
else:
    print("❌ Font file not found. Re-run Step 3.")

## Example 1 — Bengali Line Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = [1, 2, 3, 4, 5]
y = [2, 4, 3, 5, 4]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y, marker="o", color="steelblue")
ax.grid(True, alpha=0.3)

br.set_bangla_title(ax,  "বাংলা লাইন প্লট",  font_size=28)
br.set_bangla_xlabel(ax, "সময় (মাস)",         font_size=22)
br.set_bangla_ylabel(ax, "মান",                font_size=22)
br.set_bangla_xticks(ax, x,
                     ["জানু", "ফেব্রু", "মার্চ", "এপ্রিল", "মে"],
                     font_size=16, zoom=0.38,
                     collision_avoidance=False)
br.set_bangla_yticks(ax, [2, 3, 4, 5],
                     ["২", "৩", "৪", "৫"],
                     font_size=16, zoom=0.38,
                     collision_avoidance=False)

br.apply_bangla_layout(fig, auto=True)
plt.savefig("line_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Line plot saved.")

## Example 2 — Bengali Heatmap

In [ ]:
data = np.array([
    [0.1, 0.5, 0.9],
    [0.3, 0.7, 0.4],
    [0.8, 0.2, 0.6],
])

cells = [
    ["খুশি",   "দুঃখ",    "রাগ"],
    ["ভয়",    "আশা",     "বিশ্বাস"],
    ["শান্তি", "ঘৃণা",    "আনন্দ"],
]

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(data, cmap="viridis", origin="upper", aspect="equal")

rows, cols = data.shape
for i in range(rows):
    for j in range(cols):
        br.add_bangla_in_cell(ax, i, j, cells[i][j],
                              rows, cols, font_size=22)

br.set_bangla_title(ax,  "বাংলা ইমোশন হিটম্যাপ", font_size=28)
br.set_bangla_xlabel(ax, "পূর্বাভাস শ্রেণি",      font_size=22)
br.set_bangla_ylabel(ax, "আসল শ্রেণি",             font_size=22)
br.set_bangla_xticks(ax, [0, 1, 2],
                     ["প্রথম", "দ্বিতীয়", "তৃতীয়"],
                     font_size=16, zoom=0.38,
                     collision_avoidance=False)
br.set_bangla_yticks(ax, [0, 1, 2],
                     ["এক", "দুই", "তিন"],
                     font_size=16, zoom=0.38,
                     collision_avoidance=False)

fig.colorbar(im, ax=ax).ax.set_ylabel("Value", rotation=-90, va="bottom")
br.apply_bangla_layout(fig, auto=True)
plt.savefig("heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Heatmap saved.")

## Example 3 — Bengali Confusion Matrix

In [ ]:
cm = np.array([
    [863,  1343, 193],
    [585,  3710, 541],
    [26,   245,  7003],
])

classes = ["ঘৃণা", "অপমানজনক", "ঠিক আছে"]

fig, ax = plt.subplots(figsize=(7, 6.5))
im = ax.imshow(cm, cmap="autumn", origin="upper", aspect="equal")

rows, cols = cm.shape
ax.set_xlim(-0.5, cols - 0.5)
ax.set_ylim(rows - 0.5, -0.5)

for i in range(rows):
    for j in range(cols):
        ax.text(j, i, str(cm[i, j]),
                ha="center", va="center",
                color="black", fontsize=14, fontweight="bold")

br.set_bangla_title(ax,  "কনফিউশন ম্যাট্রিক্স", font_size=28)
br.set_bangla_xlabel(ax, "পূর্বাভাস শ্রেণি",     font_size=22)
br.set_bangla_ylabel(ax, "আসল শ্রেণি",            font_size=22)
br.set_bangla_xticks(ax, list(range(cols)), classes,
                     font_size=16, zoom=0.40,
                     collision_avoidance=False)
br.set_bangla_yticks(ax, list(range(rows)), classes,
                     font_size=16, zoom=0.40,
                     collision_avoidance=False)

fig.colorbar(im, ax=ax).ax.set_ylabel(
    "Normalized value", rotation=-90, va="bottom")

br.apply_bangla_layout(fig, auto=True)
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved.")

## Example 4 — Complex Bengali Words

These are words that **break completely** in default Matplotlib but render correctly with bangla-render.

In [ ]:
words = [
    "দৃষ্টিভঙ্গি",
    "ব্যবস্থাপনা",
    "হাস্যোজ্জ্বল",
    "পর্যালোচনায়",
    "জ্ঞানোদয়",
    "শ্রদ্ধা",
]

fig, ax = plt.subplots(figsize=(6, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

for idx, word in enumerate(words):
    y_pos = 0.9 - idx * 0.14
    br.bangla_text(ax, 0.5, y_pos, word,
                   coord="axes", ha="center", va="center",
                   font_size=28)

br.set_bangla_title(ax, "জটিল বাংলা শব্দ", font_size=26)
br.apply_bangla_layout(fig, auto=True)
plt.savefig("complex_words.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Complex words saved.")

---

## ✅ Summary

| Feature | Status |
|---|---|
| Headless Qt (offscreen) | ✅ |
| Bengali font registration | ✅ |
| Line plot with Bengali labels | ✅ |
| Heatmap with Bengali cell text | ✅ |
| Confusion matrix | ✅ |
| Complex conjunct rendering | ✅ |

**bangla-render** works on Kaggle and Google Colab with font registration.

```
pip install bangla-render
```

GitHub: https://github.com/mbs57/bangla-render  
PyPI: https://pypi.org/project/bangla-render/

---